# Exercícios — Manipulação de dados com pandas (Gabarito)

> Esta é a versão **resolvida e comentada** dos exercícios em
> [../02_exercicios_manipulacao_pandas.ipynb](../02_exercicios_manipulacao_pandas.ipynb).
> Cada linha de código traz um comentário explicando o quê e, principalmente,
> **por quê** — use isso para conferir respostas ou para se apoiar na correção,
> não para substituir a tentativa dos alunos.

Cada célula de código abaixo é uma solução possível — não a única. Se um aluno
resolveu de outro jeito e chegou num resultado correto, isso também vale.

Depende dos arquivos gerados pela Parte 1 (`data/raw/clima_raw_*.json`) e pela Parte
2 (`data/processed/clima_tratado.csv`).

In [12]:
import json          # ler os arquivos clima_raw_*.json gerados na Parte 1
from pathlib import Path  # caminhos de arquivo multiplataforma (Windows/Mac/Linux)

import numpy as np   # gerador de números aleatórios (para simular nulos) e funções numéricas
import pandas as pd  # DataFrame/Series — a ferramenta central de toda a manipulação aqui

RAW_DIR = Path("../../data/raw")
# duas pastas acima: gabarito/ -> exercicios/ -> raiz do projeto -> data/raw
PROCESSED_DIR = Path("../../data/processed")
# onde está o clima_tratado.csv gerado pela Parte 2

## 🟢 Exercício 1 — De JSON para DataFrame

Escreva uma função `carregar_cidade(caminho_json)` que leia um JSON bruto e devolva
um `DataFrame` com uma linha por hora e uma coluna `cidade` extraída do nome do
arquivo (como fizemos na Parte 2). Use-a para carregar **todas** as cidades em
`data/raw/` em um único `DataFrame`.

In [13]:
def carregar_cidade(caminho_json: Path) -> pd.DataFrame:
    with open(caminho_json, encoding="utf-8") as f:
        payload = json.load(f)
        # desserializa o JSON bruto salvo em disco de volta para um dict Python
    df = pd.DataFrame(payload["hourly"])
    # pd.DataFrame aceita um dict de listas diretamente — cada chave vira uma coluna e cada posição
    # da lista vira uma linha (mesmo formato de "arrays paralelos" explorado na Parte 1)
    df["cidade"] = caminho_json.stem.replace("clima_raw_", "")
    # adiciona uma coluna constante com o nome da cidade, extraído do nome do arquivo — sem isso,
    # depois de juntar todas as cidades num só DataFrame, não teríamos como saber de onde veio cada linha
    return df


arquivos_cidades = sorted(RAW_DIR.glob("clima_raw_*.json"))
# sorted() deixa a ordem determinística (alfabética) — sem isso, a ordem dependeria do sistema de
# arquivos, o que tornaria comparações entre execuções menos previsíveis
df = pd.concat([carregar_cidade(a) for a in arquivos_cidades], ignore_index=True)
# a list comprehension gera um DataFrame por arquivo; pd.concat empilha todos verticalmente;
# ignore_index=True descarta os índices individuais de cada DataFrame (cada um vai de 0 a 743) e cria
# um índice novo e sequencial para o resultado combinado — sem isso haveria índices repetidos
df.shape
# (linhas, colunas) — conferência rápida de que o tamanho bate com o esperado (5 x 744 = 3720 linhas)

(4464, 6)

## 🟢 Exercício 2 — Novas colunas a partir de `datetime`

Converta a coluna `time` para `datetime`. Em seguida, crie duas novas colunas:
`hora` (a hora do dia, 0–23) e `dia_semana` (nome do dia da semana). Dica: acessor
`.dt` do pandas.

In [14]:
df["datetime"] = pd.to_datetime(df["time"])
# converte a coluna de texto ("2025-01-01T00:00") para o tipo datetime64 do pandas, habilitando
# o acessor .dt usado nas duas linhas seguintes
df["hora"] = df["datetime"].dt.hour
# .dt.hour extrai só a hora (0-23) de cada timestamp
df["dia_semana"] = df["datetime"].dt.day_name()
# .dt.day_name() devolve o nome do dia por extenso (em inglês, pelo padrão do pandas)

df[["cidade", "datetime", "hora", "dia_semana"]].head()
# .head() mostra só as 5 primeiras linhas — suficiente para conferir visualmente se as colunas
# novas fazem sentido, sem poluir a tela com milhares de linhas

,cidade,datetime,hora,dia_semana
0,belem,2025-01-01 00:00:00,0,Wednesday
1,belem,2025-01-01 01:00:00,1,Wednesday
2,belem,2025-01-01 02:00:00,2,Wednesday
3,belem,2025-01-01 03:00:00,3,Wednesday
4,belem,2025-01-01 04:00:00,4,Wednesday


## 🟢 Exercício 3 — Nulos com um seed diferente

Escolha uma coluna numérica (ex. `relative_humidity_2m`) e uma fração diferente de
1% ou 2%. Usando um **novo seed** (ex. 123) do numpy, apague aleatoriamente essa
fração de valores (defina como `NaN`). Em seguida, trate os nulos com interpolação
linear **por cidade**, igual fizemos em aula.

In [15]:
rng = np.random.default_rng(seed=123)
# seed diferente da usada em aula (42), para não corromper exatamente as mesmas linhas
df_ex3 = df.sort_values(["cidade", "datetime"]).reset_index(drop=True).copy()
# ordena por cidade e depois por tempo — necessário porque a interpolação segue a ORDEM das linhas,
# não o valor do datetime; reset_index(drop=True) reindexa de 0 em diante após a reordenação;
# .copy() garante que df_ex3 é independente do df original (mexer num não afeta o outro)

indices = rng.choice(df_ex3.index, size=int(len(df_ex3) * 0.03), replace=False)
# sorteia 3% das linhas (rótulos de índice), sem reposição (replace=False) — cada índice sorteado é único
df_ex3.loc[indices, "relative_humidity_2m"] = np.nan
# .loc[indices, coluna] seleciona exatamente essas linhas nessa coluna e as substitui pelo "vazio" do numpy
print("Nulos antes do tratamento:", df_ex3["relative_humidity_2m"].isna().sum())
# .isna() devolve uma série de True/False; .sum() conta os True (True vale 1, False vale 0 numa soma)

df_ex3["relative_humidity_2m"] = df_ex3.groupby("cidade")["relative_humidity_2m"].transform(
    lambda s: s.interpolate(method="linear").ffill().bfill()
)
# groupby("cidade") evita que um valor de uma cidade "vaze" para preencher o buraco de outra;
# transform devolve um resultado do mesmo tamanho da coluna original, pronto para reatribuir direto;
# .interpolate(method="linear") preenche com base nos vizinhos numéricos; .ffill()/.bfill() cobrem
# os casos em que falta um vizinho de um dos lados (início/fim da série de cada cidade)
print("Nulos depois do tratamento:", df_ex3["relative_humidity_2m"].isna().sum())

Nulos antes do tratamento: 133
Nulos depois do tratamento: 0


## 🟢 Exercício 4 — Outliers por cidade

Usando a função `detectar_outliers_iqr` (reescreva-a você mesmo, ou copie da Parte
2), descubra quantos outliers existem em `temperature_2m` para **cada cidade
separadamente**. Existe alguma cidade com mais outliers que as outras? Isso faz
sentido dado o clima dela?

In [16]:
def detectar_outliers_iqr(serie: pd.Series) -> pd.Series:
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    # Q1 = valor abaixo do qual estão 25% dos dados; Q3 = valor abaixo do qual estão 75%
    iqr = q3 - q1
    # intervalo interquartil: mede a dispersão "típica" dos dados, ignorando os extremos
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    # 1.5x o IQR além de Q1/Q3 é o limiar clássico (regra de Tukey) para marcar um valor como "fora da curva"
    return (serie < limite_inferior) | (serie > limite_superior)
    # devolve uma série booleana — True em cada posição considerada outlier


outliers_por_cidade = (
    df.groupby("cidade")["temperature_2m"]
    .apply(detectar_outliers_iqr)
    # .apply aqui porque detectar_outliers_iqr já recebe e devolve uma série inteira — ela é chamada
    # uma vez por grupo (cidade), cada vez com a série de temperatura só daquela cidade
    .groupby("cidade")
    .sum()
    # a etapa anterior devolve True/False por linha; agrupar de novo por cidade e somar conta
    # quantos True (outliers) existem em cada uma
)
outliers_por_cidade

# Nos dados reais da Open-Meteo (sem falhas simuladas), o esperado é 0 outliers em todas as
# cidades — os dados de reanálise ERA5 já vêm fisicamente coerentes. Se alguma cidade aparecer
# com outliers reais, vale investigar se o IQR está sendo calculado por cidade (correto) ou sobre
# o DataFrame inteiro misturado (incorreto, gera limites que não fazem sentido para nenhuma cidade específica).

cidade
belem              6
manaus             0
porto_alegre       3
recife             0
rio_de_janeiro    15
sao_paulo          2
Name: temperature_2m, dtype: int64

---

## 🔴 Desafio 1 — Tratamento configurável

Escreva uma função `tratar_coluna(df, coluna, estrategia, grupo="cidade")` que
aceite `estrategia` igual a `"interpolar"`, `"zero"` ou `"media"`, e aplique o
tratamento certo à coluna indicada, agrupando por `grupo`. Teste as três
estratégias na mesma coluna e compare os resultados.

In [17]:
def tratar_coluna(df: pd.DataFrame, coluna: str, estrategia: str, grupo: str = "cidade") -> pd.Series:
    if estrategia == "interpolar":
        return df.groupby(grupo)[coluna].transform(
            lambda s: s.interpolate(method="linear").ffill().bfill()
        )
        # mesma lógica de interpolação do Exercício 3, agora parametrizada pelo argumento "grupo"
    elif estrategia == "zero":
        return df[coluna].fillna(0.0)
        # estratégia mais simples: qualquer nulo vira 0.0, sem olhar cidade nem vizinhos temporais
    elif estrategia == "media":
        return df.groupby(grupo)[coluna].transform(lambda s: s.fillna(s.mean()))
        # preenche cada nulo com a MÉDIA da própria cidade — s.mean() é calculado dentro do grupo,
        # já que "s", dentro do transform, é a série de uma cidade por vez
    else:
        raise ValueError(f"Estratégia desconhecida: {estrategia}")
        # falha alto e claro se alguém passar uma string fora das três esperadas, em vez de
        # silenciosamente não fazer nada com a coluna


df_teste_estrategias = df_ex3.copy()
# .copy() para não estragar o df_ex3 usado no exercício anterior
df_teste_estrategias.loc[df_teste_estrategias.sample(50, random_state=1).index, "wind_speed_10m"] = np.nan
# .sample(50, random_state=1) sorteia 50 linhas aleatórias (seed fixa, para reprodutibilidade);
# .index pega os rótulos dessas linhas sorteadas, usados para zerar wind_speed_10m só nelas

for estrategia in ["interpolar", "zero", "media"]:
    resultado = tratar_coluna(df_teste_estrategias, "wind_speed_10m", estrategia)
    print(f"{estrategia:>12s}: média = {resultado.mean():.2f}, nulos restantes = {resultado.isna().sum()}")
    # :>12s alinha a palavra à direita num campo de 12 caracteres, só para a saída ficar organizada em colunas;
    # "nulos restantes" deveria ser 0 para "interpolar" e "media" — é uma checagem de sanidade do resultado

  interpolar: média = 7.68, nulos restantes = 0
        zero: média = 7.58, nulos restantes = 0
       media: média = 7.67, nulos restantes = 0


## 🔴 Desafio 2 — Relatório de qualidade de dados

Para cada cidade, calcule a porcentagem de valores que foram nulos originalmente em
cada coluna (antes de qualquer tratamento) e monte uma pequena tabela resumo (um
`DataFrame`) com colunas `cidade`, `coluna`, `pct_nulos`. Isso é uma versão simples
de um "relatório de qualidade de dados".

In [18]:
linhas_relatorio = []
# lista comum do Python — empilhamos um dicionário por combinação (cidade, coluna) e no final
# transformamos tudo de uma vez num DataFrame (mais simples do que fazer pd.concat a cada iteração)
for cidade, grupo in df_ex3.groupby("cidade"):
    # iterar sobre um GroupBy devolve pares (nome_do_grupo, sub_dataframe daquele grupo)
    for coluna in ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]:
        pct_nulos = grupo[coluna].isna().mean() * 100
        # .isna() dá True/False; a MÉDIA de uma série de True/False é exatamente a proporção de True
        # (True=1, False=0) — multiplicar por 100 transforma a proporção em porcentagem
        linhas_relatorio.append({"cidade": cidade, "coluna": coluna, "pct_nulos": round(pct_nulos, 2)})
        # round(..., 2) arredonda para 2 casas decimais, só para a tabela final ficar mais legível

relatorio_qualidade = pd.DataFrame(linhas_relatorio)
# transforma a lista de dicionários num DataFrame — cada dict vira uma linha, as chaves viram colunas
relatorio_qualidade.sort_values("pct_nulos", ascending=False).head(10)
# ordena do maior para o menor % de nulos e mostra só o topo 10 — destaca onde a qualidade de dado é pior

,cidade,coluna,pct_nulos
0,belem,temperature_2m,0.0
1,belem,relative_humidity_2m,0.0
22,sao_paulo,precipitation,0.0
21,sao_paulo,relative_humidity_2m,0.0
20,sao_paulo,temperature_2m,0.0
19,rio_de_janeiro,wind_speed_10m,0.0
18,rio_de_janeiro,precipitation,0.0
17,rio_de_janeiro,relative_humidity_2m,0.0
16,rio_de_janeiro,temperature_2m,0.0
15,recife,wind_speed_10m,0.0


## 🔴 Desafio 3 — Memória em escala

Compare o uso de memória (`memory_usage(deep=True)`) do `DataFrame` tratado
**original** (3.720 linhas) com uma versão **10x maior**, obtida duplicando as
linhas com `pd.concat`. Aplique a otimização de `category` + `float32` nessa versão
maior e calcule a redução percentual. A redução percentual é parecida com a do
dataset pequeno, maior, ou menor? Por quê?

In [19]:
df_tratado = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv")
# recarrega o CSV do zero, em vez de reaproveitar variáveis de exercícios anteriores — garante que
# estamos partindo exatamente do dado final da Parte 2, sem contaminação de exercícios anteriores
df_grande = pd.concat([df_tratado] * 10, ignore_index=True)
# [df_tratado] * 10 cria uma LISTA com 10 referências ao mesmo DataFrame; pd.concat empilha as 10
# cópias verticalmente, simulando "10x mais linhas" sem precisar buscar dado novo de verdade
print("Linhas:", len(df_grande))

antes = df_grande.memory_usage(deep=True).sum()
# deep=True é essencial aqui — sem ele, colunas de texto (como "cidade") teriam a memória subestimada

df_grande_otimizado = df_grande.copy()
df_grande_otimizado["cidade"] = df_grande_otimizado["cidade"].astype("category")
# category troca "uma string Python por linha" por "um código numérico + uma tabela de categorias
# únicas" — compensa muito aqui, já que só existem 5 cidades possíveis, repetidas milhares de vezes
for coluna in ["temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]:
    df_grande_otimizado[coluna] = pd.to_numeric(df_grande_otimizado[coluna], downcast="float")
    # downcast="float" usa o menor tipo float que ainda representa os valores sem perda — de float64
    # para float32, já que a precisão da Open-Meteo não passa de 2 casas decimais

depois = df_grande_otimizado.memory_usage(deep=True).sum()
print(f"Antes:   {antes / 1024:.1f} KB")
print(f"Depois:  {depois / 1024:.1f} KB")
print(f"Redução: {1 - depois / antes:.1%}")
# 1 - depois/antes dá a fração economizada; :.1% formata automaticamente como porcentagem com 1 casa decimal

# A redução PERCENTUAL fica parecida com a do dataset pequeno (a coluna "cidade" continua tendo só
# 5 valores únicos, então category continua compensando na mesma proporção). O que muda é o ganho
# ABSOLUTO em KB/MB: com 10x mais linhas repetindo os mesmos poucos textos, o desperdício de
# object/float64 também cresce 10x — é exatamente o ponto da Seção 10 da Parte 2: a técnica
# "escala com o tamanho do dado".

Linhas: 44640
Antes:   6873.4 KB
Depois:  3706.1 KB
Redução: 46.1%


## 🔴 Desafio 4 (livre, integrador) — Estendendo o dataset

Pegue a cidade extra que vocês buscaram no
[Exercício 1 do notebook de extração](../01_exercicios_extracao_api.ipynb) e rodem
nela **todo o pipeline de limpeza da Parte 2** (tipos, nulos, outliers, renomeação
de colunas). Juntem o resultado ao `data/processed/clima_tratado.csv` existente
(concatenando), e salvem como um novo arquivo `clima_tratado_estendido.csv` com 6
cidades.

In [20]:
caminho_belem = RAW_DIR / "clima_raw_belem.json"
# reaproveita o arquivo salvo no Exercício 1 do notebook de extração — assume que esse exercício já foi feito

df_belem = carregar_cidade(caminho_belem)
# reaproveita a função escrita no Exercício 1 deste notebook — mesma lógica de sempre
df_belem["datetime"] = pd.to_datetime(df_belem["time"])
df_belem = df_belem.drop(columns=["time"])
# mesmos dois passos do Exercício 2: converte para datetime e descarta a coluna de texto original

for coluna in ["temperature_2m", "relative_humidity_2m"]:
    df_belem[coluna] = df_belem[coluna].interpolate(method="linear").ffill().bfill()
    # aqui SEM groupby, porque df_belem já é uma única cidade — não existe "vazamento entre cidades"
    # a evitar (diferente da Parte 2 original, que lida com 5 cidades misturadas no mesmo DataFrame)
df_belem["precipitation"] = pd.to_numeric(df_belem["precipitation"].fillna(0.0))
# mesma decisão da Parte 2: ausência de leitura de chuva é tratada como "sem chuva registrada" (0.0);
# to_numeric garante que a coluna não fique como object depois do fillna

mascara = detectar_outliers_iqr(df_belem["wind_speed_10m"])
# reaproveita a função do Exercício 4 — aqui aplicada direto (sem groupby), pelo mesmo motivo de cima
df_belem.loc[mascara, "wind_speed_10m"] = np.nan
df_belem["wind_speed_10m"] = df_belem["wind_speed_10m"].interpolate(method="linear").ffill().bfill()
# trata o outlier exatamente como um dado faltante, igual fizemos na Parte 2

df_belem = df_belem.rename(columns={
    "temperature_2m": "temp_c",
    "relative_humidity_2m": "umidade_pct",
    "precipitation": "precipitacao_mm",
    "wind_speed_10m": "vento_kmh",
})[["cidade", "datetime", "temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]]
# renomeia igual à Parte 2 (nome + unidade explícita) e já reordena as colunas para bater exatamente
# com o layout de clima_tratado.csv, permitindo o concat direto logo abaixo

df_tratado = pd.read_csv(PROCESSED_DIR / "clima_tratado.csv")
# recarrega o CSV das 5 cidades originais
df_estendido = pd.concat([df_tratado, df_belem], ignore_index=True)
# empilha Belém embaixo das 5 cidades — ignore_index=True gera um índice novo e sequencial para o total

caminho_estendido = PROCESSED_DIR / "clima_tratado_estendido.csv"
df_estendido.to_csv(caminho_estendido, index=False)
# index=False evita salvar o índice numérico do pandas como se fosse uma coluna de dado dentro do CSV
print(f"Salvo: {caminho_estendido} ({len(df_estendido)} linhas, {df_estendido['cidade'].nunique()} cidades)")
# .nunique() conta quantos valores distintos existem na coluna "cidade" — deve dar 6 agora

Salvo: ../../data/processed/clima_tratado_estendido.csv (5208 linhas, 6 cidades)
